
# Data Product Documentation: gold_posts_users

## Overview
`gold_posts_users` is a One Big Table (OBT) that combines post data with user information into a single dataset.

Instead of keeping posts and users separate, this table joins them once in the pipeline so that downstream users can query everything in one place.

Each row = a post enriched with its author info.

---

## Purpose
This table is designed to:

- Simplify analytics by avoiding repeated joins
- Improve performance for dashboards and queries
- Provide a consistent, ready-to-use dataset
- Centralize business logic in the data platform

---

## Data Model

### Grain
- One row per post (`PostId`)

So:

posts = main entity

users = dimension (author info)

### Structure
- **Post fields** → describe the content (title, score, tags, etc.)
- **User fields** → describe the author (reputation, name, etc.)

This makes it easy to analyze posts together with their authors.

### Source Tables
- `default.stg_posts` → post data (fact table)
- `default.users` → user data (dimension table)

### Join Logic
The table is built using a LEFT JOIN:

`posts.OwnerUserId = users.Id`

---

## Example Output

| PostId | Title | Score | UserId | DisplayName | Reputation |
|--------|------|-------|--------|-------------|------------|
| 53 | null | 65 | 28 | miku | 860 |
| 76 | null | 9 | 28 | miku | 860 |

Notes:
- Each row represents a post
- User information is attached to each post
- Some fields (e.g., Title) may be null depending on PostType

In [0]:
df_stg_posts = spark.read.table('`data-plataform-jayzern`.default.stg_posts')
display(df_stg_posts.limit(3))

In [0]:
df_users = spark.read.table('`data-plataform-jayzern`.default.users')
display(df_users.limit(3))

In [0]:
%sql

CREATE OR REPLACE TABLE `data-plataform-jayzern`.default.marts_posts_users AS
SELECT
    -- =====================
    -- POST FIELDS (fact side)
    -- =====================
    p.PostId,
    p.PostType,
    p.ParentId,
    p.CreationDate AS PostCreationDate,
    p.Score,
    p.ViewCount,
    p.Title,
    p.TagsArray,
    p.AnswerCount,
    p.CommentCount,

    -- =====================
    -- USER FIELDS (dimension side)
    -- =====================
    u.Id AS UserId,
    u.DisplayName,
    u.Reputation,
    u.UpVotes,
    u.DownVotes,
    u.Views AS UserViews,
    u.CreationDate AS UserCreationDate,
    u.LastAccessDate,
    u.Location,
    u.WebsiteUrl

FROM `data-plataform-jayzern`.default.stg_posts p
LEFT JOIN `data-plataform-jayzern`.default.users u
    ON p.OwnerUserId = u.Id;

In [0]:
%sql
SELECT *
FROM `data-plataform-jayzern`.default.marts_posts_users
LIMIT 5;